In [1]:
import pandas as pd, numpy as np
from pathlib import Path
inp, W = Path("/kaggle/input"), Path("/kaggle/working")

det = pd.read_csv(next(inp.rglob("count_detections.csv")), dtype={"unit": str})
tz = next(p for p in inp.rglob("tz_split") if p.is_dir())
tz_lab = pd.read_csv(next(inp.rglob("tz_labels.csv")), dtype={"image": str})
tile2photo = tz_lab.drop_duplicates("image").set_index("image").photo
gh_split = pd.read_csv(next(inp.rglob("gh_image_split.csv")), index_col=0).split
gh_lab = pd.read_csv(next(inp.rglob("gh_labels.csv")), dtype={"image": str})
TRUE = {"tz": tz_lab[tz_lab.cls == "parasite"].groupby("photo").size(),
        "gh": gh_lab[gh_lab.cls == 0].groupby("image").size()}
MODELS = {"tz": ["never seen (F)", "50 local images", "home (C)"],
          "gh": ["never seen (E)", "50 local images", "home (B)"]}
U = {"tz": {sp: sorted({tile2photo[p.stem.removeprefix("tz_")] for p in (tz / sp / "images").glob("*.jpg")}) for sp in ["val", "test"]},
     "gh": {sp: [s.removeprefix("gh_") for s, v in gh_split.items() if v == sp] for sp in ["val", "test"]}}

def counts(site, model, sp, t):
    d = det[(det.site == site) & (det.model == model) & (det.split == sp) & (det.cls == 0) & (det.conf >= t)]
    return (d.groupby("unit").size().reindex(U[site][sp], fill_value=0),
            TRUE[site].reindex(U[site][sp], fill_value=0))

def err(pred, true):
    m = true > 0; r = pred[m] / true[m]
    return np.median(np.abs(r - 1)) * 100, (np.abs(r - 1) <= 0.25).mean() * 100, pred.sum() / true.sum()

out = []
for site in ["tz", "gh"]:
    for model in MODELS[site]:
        grid = np.round(np.arange(0.05, 0.76, 0.05), 2)
        t_val = grid[np.argmin([err(*counts(site, model, "val", t))[0] for t in grid])]   
        for mode, t in [("fixed 0.25", 0.25), ("tuned on local val", t_val)]:
            pred, true = counts(site, model, "test", t)
            e = err(pred, true)
            m = true > 0; r = (pred[m] / true[m]).to_numpy()
            boot = [np.median(np.abs(np.random.default_rng(i).choice(r, len(r)) - 1)) * 100 for i in range(1000)]
            out.append(dict(site=site, model=model, cutoff=mode, t=t, n=int(m.sum()),
                            median_abs_err=round(e[0], 1), ci=f"{np.percentile(boot, 2.5):.0f}-{np.percentile(boot, 97.5):.0f}",
                            within_25pct=round(e[1]), total_ratio=round(e[2], 2)))
res = pd.DataFrame(out)
res.to_csv(W / "count_results.csv", index=False)
print(res.to_string(index=False))

site           model             cutoff    t   n  median_abs_err      ci  within_25pct  total_ratio
  tz  never seen (F)         fixed 0.25 0.25 111           203.7 172-282             0         2.80
  tz  never seen (F) tuned on local val 0.75 111            58.3   32-82            36         1.26
  tz 50 local images         fixed 0.25 0.25 111            25.0   20-32            51         1.25
  tz 50 local images tuned on local val 0.35 111            21.4   17-27            56         1.03
  tz        home (C)         fixed 0.25 0.25 111            22.2   18-25            61         1.12
  tz        home (C) tuned on local val 0.25 111            22.2   18-25            61         1.12
  gh  never seen (E)         fixed 0.25 0.25 598            94.5   93-95             0         0.17
  gh  never seen (E) tuned on local val 0.05 598            63.3   60-67            14         0.68
  gh 50 local images         fixed 0.25 0.25 598            12.0   11-13            74         1.01


In [2]:
import cv2
from scipy.stats import spearmanr

def feats(path):
    im = cv2.imread(str(path)); g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY); m = g > 40          
    lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB)
    return cv2.Laplacian(g, cv2.CV_64F)[m].var(), g[m].mean(), lab[..., 1][m].mean(), lab[..., 2][m].mean()
cols = ["focus", "brightness", "colour_a (green-red)", "colour_b (blue-yellow)"]

tz_f = pd.DataFrame([(tile2photo[p.stem.removeprefix("tz_")], *feats(p)) for p in (tz / "test" / "images").glob("*.jpg")],
                    columns=["unit"] + cols).groupby("unit").mean()
gh_img = next(p for p in inp.rglob("gh_images") if p.is_dir()) / "test" / "images"
gh_f = pd.DataFrame([(u, *feats(gh_img / f"gh_{u}.jpg")) for u in U["gh"]["test"]], columns=["unit"] + cols).set_index("unit")
F = {"tz": tz_f, "gh": gh_f}

rows = []
for site in ["tz", "gh"]:
    for model in MODELS[site]:
        t = res[(res.site == site) & (res.model == model) & (res.cutoff == "tuned on local val")].t.iloc[0]
        pred, true = counts(site, model, "test", t)
        d = F[site].join(pd.DataFrame({"abs_err": (pred / true - 1).abs()})[true > 0], how="inner")
        d["focus_third"] = pd.qcut(d.focus, 3, labels=["blurriest", "middle", "sharpest"])
        by = d.groupby("focus_third", observed=True).abs_err.median().mul(100).round(0)
        rows.append(dict(site=site, model=model, **{f"err% {k}": v for k, v in by.items()},
                         **{f"rho {c.split(' ')[0]}": round(spearmanr(d[c], d.abs_err)[0], 2) for c in cols}))
q = pd.DataFrame(rows)
q.to_csv(W / "quality_vs_error.csv", index=False)
print(q.to_string(index=False))
print("\nFocus range per site (median, 10th-90th percentile):")
for s in F: print(s, F[s].focus.median().round(0), F[s].focus.quantile([.1, .9]).round(0).tolist())

site           model  err% blurriest  err% middle  err% sharpest  rho focus  rho brightness  rho colour_a  rho colour_b
  tz  never seen (F)           153.0         29.0           30.0      -0.32           -0.44         -0.33         -0.39
  tz 50 local images            19.0         27.0           20.0       0.07            0.17          0.19          0.20
  tz        home (C)            12.0         27.0           23.0       0.25            0.37          0.03          0.04
  gh  never seen (E)            57.0         66.0           71.0       0.07            0.01          0.14         -0.19
  gh 50 local images            11.0         15.0           11.0       0.01            0.05          0.19         -0.26
  gh        home (B)             8.0         11.0            9.0       0.01           -0.02          0.23         -0.19

Focus range per site (median, 10th-90th percentile):


AttributeError: 'float' object has no attribute 'round'

In [3]:
print("Focus range per site (median, 10th-90th percentile):")
for s in F: print(s, round(F[s].focus.median()), F[s].focus.quantile([.1, .9]).round(0).tolist())

Focus range per site (median, 10th-90th percentile):
tz 151 [83.0, 189.0]
gh 11 [5.0, 30.0]
